# NHS API integration tests

`src/test_apis.py` - URLs, API-key headers, retry/backoff rules, and integration pacing.

A failed NHS Website Content request will not prevent the Directory of Healthcare Services checks from running.


In [1]:
import math
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

current = Path.cwd().resolve()
if (current / "src" / "test_apis.py").exists():
    PROJECT_ROOT = current
elif (current.parent / "src" / "test_apis.py").exists():
    PROJECT_ROOT = current.parent
else:
    raise RuntimeError("Open this notebook from the nhs_api_starter project or its notebooks folder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from test_apis import CONTENT_BASES, DOHS_BASES, api_key_headers, get_json

load_dotenv(PROJECT_ROOT / ".env")
ENVIRONMENT = os.getenv("NHS_ENV", "integration").strip().lower()
if ENVIRONMENT not in CONTENT_BASES:
    raise ValueError(f"Unsupported NHS_ENV: {ENVIRONMENT!r}")

CONTENT_KEY = os.getenv("NHS_CONTENT_API_KEY") or None
DOHS_KEY = os.getenv("NHS_DOHS_API_KEY") or None

configuration = pd.DataFrame([
    {"Setting": "Environment", "Value": ENVIRONMENT},
    {"Setting": "Website Content key loaded", "Value": "Yes" if CONTENT_KEY else "No"},
    {"Setting": "DoHS key loaded", "Value": "Yes" if DOHS_KEY else "No"},
    {"Setting": "Minimum integration interval", "Value": "1.05 seconds"},
])
configuration

,Setting,Value
0,Environment,integration
1,Website Content key loaded,Yes
2,DoHS key loaded,Yes
3,Minimum integration interval,1.05 seconds


The HTTP helper comes from `src/test_apis.py`, so HTTP 502, 503, and 504 responses receive the same three-attempt exponential backoff. It also spaces integration requests by at least 1.05 seconds.

In [ ]:
def run_check(name, check_function):
    try:
        details, table = check_function()
        result = {"Check": name, "Result": "PASS", "Details": details}
        print(f"[PASS] {name}: {details}")
        return result, table
    except Exception as exc:
        details = f"{type(exc).__name__}: {exc}"
        result = {"Check": name, "Result": "FAIL", "Details": details}
        print(f"[FAIL] {name}: {details}")
        return result, pd.DataFrame()


def eps_enabled(value):
    return str(value).strip().lower() == "true"


def distance_miles(latitude, longitude, origin_latitude, origin_longitude):
    earth_radius_miles = 3958.8
    delta_latitude = math.radians(latitude - origin_latitude)
    delta_longitude = math.radians(longitude - origin_longitude)
    a = (
        math.sin(delta_latitude / 2) ** 2
        + math.cos(math.radians(origin_latitude))
        * math.cos(math.radians(latitude))
        * math.sin(delta_longitude / 2) ** 2
    )
    return earth_radius_miles * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("Helpers ready checkkkkkkkkkkkkkkkkkkkkkkkkkkkk")

Helpers ready.


## 1. NHS Website Content API v2 — Symptoms A

Expected result: HTTP 200 and a non-empty list containing names and NHS URLs.

In [3]:
def check_website_content():
    body = get_json(
        f"{CONTENT_BASES[ENVIRONMENT]}/symptoms",
        params={"category": "a", "page": 1},
        headers=api_key_headers(CONTENT_KEY),
    )
    links = body.get("significantLink") or []
    if not links:
        raise AssertionError("The significantLink list is empty.")
    if any(not item.get("name") or not item.get("url") for item in links):
        raise AssertionError("At least one symptom item is missing its name or URL.")
    table = pd.DataFrame([
        {"Name": item.get("name"), "Modified": (item.get("mainEntityOfPage") or {}).get("dateModified"), "URL": item.get("url")}
        for item in links
    ])
    return f"{len(links)} symptom links returned", table

content_summary, symptoms_table = run_check("Website Content: Symptoms A", check_website_content)
symptoms_table


GET https://int.api.service.nhs.uk/nhs-website-content/symptoms?category=a&page=1
HTTP 200
[PASS] Website Content: Symptoms A: 4 symptom links returned


,Name,Modified,URL
0,"Anxiety, fear and panic",2026-07-14T12:14:43+00:00,https://int.api.service.nhs.uk/nhs-website-con...
1,Anger,2026-08-10T14:59:08+00:00,https://int.api.service.nhs.uk/nhs-website-con...
2,Anal pain,2026-04-13T09:10:11+00:00,https://www.nhs.uk/symptoms/anal-pain/
3,Ankle pain,2026-07-09T14:59:25+00:00,https://www.nhs.uk/symptoms/foot-pain/ankle-pain/


## 2. DoHS v3 — Known organisation

Expected result: HTTP 200 and ODS code `Y02494` for Shakespeare Medical Practice.

In [4]:
def check_known_organisation():
    body = get_json(
        f"{DOHS_BASES[ENVIRONMENT]}/",
        params={"api-version": "3", "search": "Y02494", "$top": "5"},
        headers=api_key_headers(DOHS_KEY),
    )
    records = body.get("value") or []
    matches = [item for item in records if item.get("ODSCode") == "Y02494"]
    if not matches:
        raise AssertionError("ODS code Y02494 was not returned.")
    table = pd.DataFrame([
        {
            "ODS code": item.get("ODSCode"),
            "Organisation": item.get("OrganisationName"),
            "Type": item.get("OrganisationType"),
            "City": item.get("City"),
            "Postcode": item.get("Postcode"),
        }
        for item in matches
    ])
    return f"{len(matches)} matching organisation returned", table

known_org_summary, known_org_table = run_check("DoHS v3: known organisation", check_known_organisation)
known_org_table


GET https://int.api.service.nhs.uk/service-search-api/?api-version=3&search=Y02494&%24top=5
HTTP 200
[PASS] DoHS v3: known organisation: 1 matching organisation returned


,ODS code,Organisation,Type,City,Postcode
0,Y02494,Shakespeare Medical Practice,GpBranch,Leeds,LS9 7TA


## 3. DoHS v3 — EPS-enabled community pharmacies

Expected result: five records, all with type `PHA`, subtype `Community`, and EPS enabled. The API currently serializes `IsEpsEnabled` as the JSON string `"true"`, which the check handles correctly.

In [5]:
PHARMACY_FILTER = "IsEpsEnabled eq 'true' and OrganisationTypeId eq 'PHA' and OrganisationSubType eq 'Community'"

def check_community_pharmacies():
    body = get_json(
        f"{DOHS_BASES[ENVIRONMENT]}/",
        params={
            "api-version": "3",
            "search": "*",
            "$top": "5",
            "$count": "true",
            "$filter": PHARMACY_FILTER,
        },
        headers=api_key_headers(DOHS_KEY),
    )
    records = body.get("value") or []
    if not records:
        raise AssertionError("No community pharmacies were returned.")
    invalid = [
        item for item in records
        if item.get("OrganisationTypeId") != "PHA"
        or item.get("OrganisationSubType") != "Community"
        or not eps_enabled(item.get("IsEpsEnabled"))
    ]
    if invalid:
        raise AssertionError(f"{len(invalid)} returned records do not match the pharmacy filter.")
    table = pd.DataFrame([
        {
            "ODS code": item.get("ODSCode"),
            "Organisation": item.get("OrganisationName"),
            "City": item.get("City"),
            "Postcode": item.get("Postcode"),
            "EPS enabled": item.get("IsEpsEnabled"),
        }
        for item in records
    ])
    total = body.get("@odata.count", "not supplied")
    return f"{len(records)} records shown; {total} total matches reported", table

pharmacy_summary, pharmacy_table = run_check("DoHS v3: community pharmacies", check_community_pharmacies)
pharmacy_table


GET https://int.api.service.nhs.uk/service-search-api/?api-version=3&search=%2A&%24top=5&%24count=true&%24filter=IsEpsEnabled+eq+%27true%27+and+OrganisationTypeId+eq+%27PHA%27+and+OrganisationSubType+eq+%27Community%27
HTTP 200
[PASS] DoHS v3: community pharmacies: 5 records shown; 10586 total matches reported


,ODS code,Organisation,City,Postcode,EPS enabled
0,FWR08,BOOTS,LAMPETER,SA48 7DX,true
1,FWF03,WELL,SWANSEA,SA5 8PG,true
2,FWP97,WELL,NEWPORT,NP11 6BW,true
3,FWD63,M W PHILLIPS CHEMISTS,CARDIFF,CF24 2LU,true
4,FWE12,WELL,HAVERFORDWEST,SA61 1QX,true


## 4. DoHS v3 — Nearest pharmacies to Manchester

default point is set as central Manchester: longitude `-2.2426`, latitude `53.4808`. The output adds an approximate distance in miles which is calculated using the Haversine formula.

In [6]:
ORIGIN_LONGITUDE = -2.2426
ORIGIN_LATITUDE = 53.4808

def check_nearest_pharmacies():
    body = get_json(
        f"{DOHS_BASES[ENVIRONMENT]}/",
        params={
            "api-version": "3",
            "search": "*",
            "$top": "5",
            "$filter": PHARMACY_FILTER,
            "$orderby": f"geo.distance(Geocode, geography'POINT({ORIGIN_LONGITUDE} {ORIGIN_LATITUDE})')",
        },
        headers=api_key_headers(DOHS_KEY),
    )
    records = body.get("value") or []
    if not records:
        raise AssertionError("No nearby pharmacies were returned.")
    rows = []
    for item in records:
        if item.get("Latitude") is None or item.get("Longitude") is None:
            raise AssertionError("A returned pharmacy is missing coordinates.")
        rows.append({
            "ODS code": item.get("ODSCode"),
            "Organisation": item.get("OrganisationName"),
            "Postcode": item.get("Postcode"),
            "Distance (miles)": round(distance_miles(item["Latitude"], item["Longitude"], ORIGIN_LATITUDE, ORIGIN_LONGITUDE), 2),
        })
    distances = [row["Distance (miles)"] for row in rows]
    if any(distances[index] + 0.01 < distances[index - 1] for index in range(1, len(distances))):
        raise AssertionError("Results are not ordered nearest first.")
    table = pd.DataFrame(rows)
    return f"{len(records)} pharmacies returned in nearest-first order", table

nearest_summary, nearest_table = run_check("DoHS v3: nearest Manchester pharmacies", check_nearest_pharmacies)
nearest_table


GET https://int.api.service.nhs.uk/service-search-api/?api-version=3&search=%2A&%24top=5&%24filter=IsEpsEnabled+eq+%27true%27+and+OrganisationTypeId+eq+%27PHA%27+and+OrganisationSubType+eq+%27Community%27&%24orderby=geo.distance%28Geocode%2C+geography%27POINT%28-2.2426+53.4808%29%27%29
HTTP 200
[PASS] DoHS v3: nearest Manchester pharmacies: 5 pharmacies returned in nearest-first order


,ODS code,Organisation,Postcode,Distance (miles)
0,FEJ12,Boots,M1 1PL,0.13
1,FXD87,SUPERDRUG PHARMACY,M1 1LZ,0.21
2,FD532,Boots,M1 1LY,0.23
3,FJG88,Cameolord Pharmacy,M1 5AE,0.28
4,FRC23,Boots,M1 4RL,0.30


All four rows should say PASS. A failure remains visible in the summary without stopping the later checks.

In [7]:
summary = pd.DataFrame([
    content_summary,
    known_org_summary,
    pharmacy_summary,
    nearest_summary,
])
passed = int((summary["Result"] == "PASS").sum())
failed = int((summary["Result"] == "FAIL").sum())
print(f"Completed: {passed} passed, {failed} failed")
summary

Completed: 4 passed, 0 failed


,Check,Result,Details
0,Website Content: Symptoms A,PASS,4 symptom links returned
1,DoHS v3: known organisation,PASS,1 matching organisation returned
2,DoHS v3: community pharmacies,PASS,5 records shown; 10586 total matches reported
3,DoHS v3: nearest Manchester pharmacies,PASS,5 pharmacies returned in nearest-first order
